# Sparkify: Data Modeling with Apache Cassandra

Sparkify's analytics team wants to understand what users listen to, but the activity data sits in a directory of daily CSV files. This notebook

1. **Part I** consolidates the daily event files into one denormalized file, `event_datafile_new.csv`.
2. **Part II** models three query-specific Apache Cassandra tables on that file, loads them, and answers the analytics team's three questions.

# Part I. ETL Pipeline for Pre-Processing the Files

#### Import Python packages

In [1]:
import csv
import glob
import os

import pandas as pd

#### Collect the raw event files
Only top-level `*.csv` files in `event_data/` are used, in sorted (date) order. Sub-directories such as `.ipynb_checkpoints` are ignored because the pattern is not recursive.

In [2]:
filepath = os.path.join(os.getcwd(), 'event_data')
file_path_list = sorted(glob.glob(os.path.join(filepath, '*.csv')))

print(f'{len(file_path_list)} event files found in {filepath}')
print(f'first: {os.path.basename(file_path_list[0])}  last: {os.path.basename(file_path_list[-1])}')

30 event files found in /Users/suleimanadebowaleojo/Claude/Projects/Data Engineering Projects with AZURE/1.Data Modeling with Apache Cassandra/event_data
first: 2018-11-01-events.csv  last: 2018-11-30-events.csv


#### Create `event_datafile_new.csv`
Rows without an artist (non-playback events such as page visits) are dropped. Columns are looked up by header name, so the output keeps the required order:
`artist, firstName, gender, itemInSession, lastName, length, level, location, sessionId, song, userId`.
The output file is opened in write mode, so re-running the cell regenerates it instead of appending duplicates.

In [3]:
OUTPUT_FILE = 'event_datafile_new.csv'
OUTPUT_COLUMNS = ['artist', 'firstName', 'gender', 'itemInSession', 'lastName', 'length',
                  'level', 'location', 'sessionId', 'song', 'userId']


def read_playback_rows(paths):
    """Yield one dict per playback event (non-empty artist) from the raw event files."""
    for path in paths:
        with open(path, 'r', encoding='utf8', newline='') as csvfile:
            for record in csv.DictReader(csvfile):
                if record['artist'] != '':
                    yield record


with open(OUTPUT_FILE, 'w', encoding='utf8', newline='') as f:
    writer = csv.writer(f, quoting=csv.QUOTE_ALL)
    writer.writerow(OUTPUT_COLUMNS)
    for record in read_playback_rows(file_path_list):
        writer.writerow([record[col] for col in OUTPUT_COLUMNS])

#### Validate the generated file

In [4]:
assert os.path.exists(OUTPUT_FILE), 'event_datafile_new.csv was not created'

with open(OUTPUT_FILE, 'r', encoding='utf8', newline='') as f:
    rows = list(csv.reader(f))

header, data = rows[0], rows[1:]
assert header == OUTPUT_COLUMNS, 'unexpected header'
assert all(len(r) == len(OUTPUT_COLUMNS) for r in data), 'row with unexpected field count'

print(f'Records (excluding header): {len(data)}')
print(f'Columns ({len(header)}): {header}')

events_df = pd.read_csv(OUTPUT_FILE, encoding='utf8')
events_df[['artist', 'song', 'length', 'level', 'sessionId', 'itemInSession']].head(3)

Records (excluding header): 6820
Columns (11): ['artist', 'firstName', 'gender', 'itemInSession', 'lastName', 'length', 'level', 'location', 'sessionId', 'song', 'userId']


,artist,song,length,level,sessionId,itemInSession
0,Des'ree,You Gotta Be,246.30812,free,139,1
1,Mr Oizo,Flat 55,144.03873,free,139,3
2,Tamba Trio,Quem Quiser Encontrar O Amor,177.18812,free,139,4


# Part II. Apache Cassandra

Cassandra tables are designed around the **queries** they must answer, not around entities. Each of the three questions gets its own denormalized table whose primary key matches the `WHERE` clause of its query, so no `ALLOW FILTERING`, index or join is needed.

#### Create the cluster connection
This connects to the Cassandra instance running on `127.0.0.1`.

In [5]:
from cassandra.cluster import Cluster

cluster = Cluster(['127.0.0.1'])
session = cluster.connect()

#### Create and set the keyspace
`SimpleStrategy` with a replication factor of 1 is appropriate for a single local node.

In [6]:
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS sparkify
    WITH REPLICATION = {'class': 'SimpleStrategy', 'replication_factor': 1}
""")
session.set_keyspace('sparkify')

#### Helpers used by all three tables

In [7]:
EVENT_FILE = 'event_datafile_new.csv'

# Position of each column in event_datafile_new.csv
COL = {name: i for i, name in enumerate(OUTPUT_COLUMNS)}


def required_text(line, name):
    value = line[COL[name]].strip()
    if value == '':
        raise ValueError(f'empty value for required column {name}: {line}')
    return value


def required_int(line, name):
    return int(required_text(line, name))


def required_float(line, name):
    return float(required_text(line, name))


def load_table(insert_cql, row_builder):
    """Insert every record of the event file with a prepared statement; return the rows read."""
    prepared = session.prepare(insert_cql)
    count = 0
    with open(EVENT_FILE, encoding='utf8') as f:
        reader = csv.reader(f)
        next(reader)  # header
        for line in reader:
            session.execute(prepared, row_builder(line))
            count += 1
    return count


def show(rows):
    return pd.DataFrame(list(rows))

## Query 1
**Give me the artist, song title and song's length in the music app history that was heard during sessionId = 338 and itemInSession = 4.**

**Table `song_by_session_item`**
- **Question served:** look up the one song played at a given position of a given session.
- **Primary key:** `PRIMARY KEY (session_id, item_in_session)`. **Partition key** `session_id` groups a session's plays on one node; **clustering column** `item_in_session` orders them inside it. Both columns appear in the `WHERE` clause with equality, so Cassandra goes straight to one row.
- **Uniqueness / grain:** one row per play event in a session. `(session_id, item_in_session)` identifies a play uniquely (verified below: no duplicate keys in the data), so no play is overwritten.

In [8]:
session.execute("DROP TABLE IF EXISTS song_by_session_item")
session.execute("""
    CREATE TABLE IF NOT EXISTS song_by_session_item (
        session_id      int,
        item_in_session int,
        artist          text,
        song            text,
        length          double,
        PRIMARY KEY (session_id, item_in_session)
    )
""")

n = load_table(
    "INSERT INTO song_by_session_item (session_id, item_in_session, artist, song, length) VALUES (?, ?, ?, ?, ?)",
    lambda l: (required_int(l, 'sessionId'), required_int(l, 'itemInSession'),
               required_text(l, 'artist'), required_text(l, 'song'), required_float(l, 'length')),
)
print(f'{n} records read from {EVENT_FILE}')

6820 records read from event_datafile_new.csv


#### Verify Query 1

In [9]:
rows = session.execute("""
    SELECT artist, song, length
    FROM song_by_session_item
    WHERE session_id = 338 AND item_in_session = 4
""")
show(rows)

,artist,song,length
0,Faithless,Music Matters (Mark Knight Dub),495.3073


## Query 2
**Give me only the following: name of artist, song (sorted by itemInSession) and user (first and last name) for userId = 10, sessionId = 182.**

**Table `song_by_user_session`**
- **Question served:** list what one user played in one session, in play order.
- **Primary key:** `PRIMARY KEY ((user_id, session_id), item_in_session)`. The **composite partition key** `(user_id, session_id)` matches the two equality filters, so all of that user's plays in that session live in one partition. The **clustering column** `item_in_session` stores them in ascending order on disk.
- **Ordering:** `WITH CLUSTERING ORDER BY (item_in_session ASC)` makes Cassandra return rows already sorted, so the query has no `ORDER BY` and nothing is sorted in Python.
- **Uniqueness / grain:** one row per play event of a user in a session. `item_in_session` is unique within a session, so the full key is unique (verified below).

In [10]:
session.execute("DROP TABLE IF EXISTS song_by_user_session")
session.execute("""
    CREATE TABLE IF NOT EXISTS song_by_user_session (
        user_id         int,
        session_id      int,
        item_in_session int,
        artist          text,
        song            text,
        first_name      text,
        last_name       text,
        PRIMARY KEY ((user_id, session_id), item_in_session)
    ) WITH CLUSTERING ORDER BY (item_in_session ASC)
""")

n = load_table(
    """INSERT INTO song_by_user_session
       (user_id, session_id, item_in_session, artist, song, first_name, last_name)
       VALUES (?, ?, ?, ?, ?, ?, ?)""",
    lambda l: (required_int(l, 'userId'), required_int(l, 'sessionId'), required_int(l, 'itemInSession'),
               required_text(l, 'artist'), required_text(l, 'song'),
               required_text(l, 'firstName'), required_text(l, 'lastName')),
)
print(f'{n} records read from {EVENT_FILE}')

6820 records read from event_datafile_new.csv


#### Verify Query 2

In [11]:
rows = session.execute("""
    SELECT artist, song, first_name, last_name
    FROM song_by_user_session
    WHERE user_id = 10 AND session_id = 182
""")
show(rows)

,artist,song,first_name,last_name
0,Down To The Bone,Keep On Keepin' On,Sylvie,Cruz
1,Three Drives,Greece 2000,Sylvie,Cruz
2,Sebastien Tellier,Kilometer,Sylvie,Cruz
3,Lonnie Gordon,Catch You Baby (Steve Pitron & Max Sanna Radio...,Sylvie,Cruz


## Query 3
**Give me every user name (first and last) in my music app history who listened to the song 'All Hands Against His Own'.**

**Table `listeners_by_song`**
- **Question served:** which users listened to a given song.
- **Primary key:** `PRIMARY KEY (song, user_id)`. **Partition key** `song` serves the `WHERE song = ...` filter; **clustering column** `user_id` makes each listener a separate row.
- **Uniqueness / grain:** one row per *distinct listener of a song*. A user who plays the same song several times is stored once, because the repeat play upserts the same `(song, user_id)` key. That is the intended behaviour: the question asks for *users*, not plays. The key must include `user_id`; with `song` alone, listeners would overwrite each other and only one name would be returned.

In [12]:
session.execute("DROP TABLE IF EXISTS listeners_by_song")
session.execute("""
    CREATE TABLE IF NOT EXISTS listeners_by_song (
        song       text,
        user_id    int,
        first_name text,
        last_name  text,
        PRIMARY KEY (song, user_id)
    )
""")

n = load_table(
    "INSERT INTO listeners_by_song (song, user_id, first_name, last_name) VALUES (?, ?, ?, ?)",
    lambda l: (required_text(l, 'song'), required_int(l, 'userId'),
               required_text(l, 'firstName'), required_text(l, 'lastName')),
)
print(f'{n} records read from {EVENT_FILE}')

6820 records read from event_datafile_new.csv


#### Verify Query 3

In [13]:
rows = session.execute("""
    SELECT first_name, last_name
    FROM listeners_by_song
    WHERE song = 'All Hands Against His Own'
""")
show(rows)

,first_name,last_name
0,Jacqueline,Lynch
1,Tegan,Levine
2,Sara,Johnson


## Data checks
Confirm that each table holds exactly one row per distinct primary key in the processed data, i.e. no play was lost to an overwrite and re-running the load adds nothing. (`COUNT(*)` scans the table and is used here only as a one-off check on this small dataset, never in the answer queries.)

In [14]:
expected = {
    'song_by_session_item': len(events_df[['sessionId', 'itemInSession']].drop_duplicates()),
    'song_by_user_session': len(events_df[['userId', 'sessionId', 'itemInSession']].drop_duplicates()),
    'listeners_by_song':    len(events_df[['song', 'userId']].drop_duplicates()),
}
checks = []
for table, expected_rows in expected.items():
    actual = list(session.execute(f'SELECT COUNT(*) FROM {table}'))[0][0]
    checks.append({'table': table, 'distinct_keys_in_csv': expected_rows,
                   'rows_in_cassandra': actual, 'match': actual == expected_rows})
checks_df = pd.DataFrame(checks)
assert checks_df['match'].all(), checks_df
print(f'Total playback records in CSV: {len(events_df)}')
checks_df

Total playback records in CSV: 6820


,table,distinct_keys_in_csv,rows_in_cassandra,match
0,song_by_session_item,6820,6820,True
1,song_by_user_session,6820,6820,True
2,listeners_by_song,6618,6618,True


## Optional cleanup: drop the project tables
Run this only after reviewing results and taking any screenshots. Set `DROP_TABLES = True` to drop the three tables created by this notebook (no other table or keyspace is touched).

In [15]:
DROP_TABLES = False

if DROP_TABLES:
    for table in ('song_by_session_item', 'song_by_user_session', 'listeners_by_song'):
        session.execute(f'DROP TABLE IF EXISTS {table}')
    print('Project tables dropped.')
else:
    print('DROP_TABLES is False: tables kept.')

DROP_TABLES is False: tables kept.


### Close the session and cluster connection

In [16]:
session.shutdown()
cluster.shutdown()